# Mission 2: The Meaning Mapper 🧠
 **Make a school assistant recognize meaning, even when the words change.** Suggested schedule: two 60-minute Friday sessions.

 **Core ideas:** a **text embedding** is a numerical representation of a sentence; **cosine similarity** compares two embeddings; **intent classification** predicts which kind of question a student is asking; **out-of-domain** means the question does not fit the topics we know.

 All questions below refer to a **fictional** school. No student data is used.

 **Before running code:** Are “Where can I check out novels?” and “How can I borrow books?” similar in meaning even if they use different words?

## 1. Load a small, pretrained language model
 *Pretrained* means people have already trained the AI model for us. We are learning how to **use and evaluate** it, not training a large AI from scratch.

 **Teacher setup:** In Google Colab, run `%pip -q install sentence-transformers` in a code cell if needed. The model needs internet to download the first time; this might take a few minutes. Use the instructor-led demonstration if student devices cannot load it.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model ready!")

## 2. Compare two questions
 The model turns a sentence into **384 numbers**. Nobody needs to memorize them. Think of those numbers as an address on a “meaning map.”

 **Prediction:** Which pair will score higher: two questions about borrowing books, or a book question and a lunch question?

In [ ]:
def similarity(text_a, text_b):
    vectors = encoder.encode([text_a, text_b], normalize_embeddings=True)
    return float(np.dot(vectors[0], vectors[1]))

pairs = [
    ("How can I borrow a book?", "Where do I check out novels?"),
    ("How can I borrow a book?", "What time is lunch?"),
]
for a, b in pairs:
    print(f"Similarity: {similarity(a,b):.2f} | {a} | {b}")

## 3. Experiment: change the meaning
 **Your turn:** Change either sentence. Try a synonym (“missing” instead of “lost”), and then a completely unrelated question. The similarity score is **not** the chance that the answer is correct.

In [ ]:
sentence_a = "Where is the lost and found?"  # CHANGE ME
sentence_b = "I misplaced my water bottle."  # CHANGE ME
print("Similarity:", round(similarity(sentence_a, sentence_b), 2))

## 4. Let the computer sort questions into topics
 An **intent** is a question's purpose. Instead of writing a complex classifier immediately, we'll use a beginner-friendly **nearest-example classifier**: choose the labeled example that is most similar to the student's question.

 This still demonstrates the **classification concept**. The original advanced notebook goes further by training logistic regression on embeddings.

In [ ]:
examples = [
  ("How do I borrow books?", "library"),
  ("Where can I check out a novel?", "library"),
  ("When does the library close?", "library"),
  ("I lost my backpack.", "lost_and_found"),
  ("Where should I look for a missing jacket?", "lost_and_found"),
  ("Can someone help find my lunchbox?", "lost_and_found"),
  ("When is lunch time?", "lunch"),
  ("What time can students eat lunch?", "lunch"),
  ("When does the cafeteria open?", "lunch"),
  ("Can I use AI on my assignment?", "ai_policy"),
  ("Does my teacher allow ChatGPT for homework?", "ai_policy"),
  ("How should I credit AI help on schoolwork?", "ai_policy"),
]
example_sentences = [sentence for sentence, intent in examples]
example_vectors = encoder.encode(example_sentences, normalize_embeddings=True)
print("We have", len(examples), "examples covering", len(set(label for _, label in examples)), "intents.")

## 5. Predict an intent
 The nearest example wins. A **threshold** makes the system say it is unsure about topics it wasn't taught.

 **Your turn:** Change `question` and `MIN_SIMILARITY`. Try “How do I repair my bicycle?”

In [ ]:
MIN_SIMILARITY = 0.45  # CHANGE ME: 0.3 is lenient; 0.7 is strict
def classify_intent(question):
    query_vector = encoder.encode([question], normalize_embeddings=True)[0]
    scores = example_vectors @ query_vector
    best = int(np.argmax(scores))
    score = float(scores[best])
    if score < MIN_SIMILARITY:
        return {"intent": "unknown", "score": score, "matched_example": examples[best][0]}
    return {"intent": examples[best][1], "score": score, "matched_example": examples[best][0]}

question = "I misplaced my notebook."  # CHANGE ME
print(classify_intent(question))

## 6. Important: an AI can still be wrong
 Different wording can make unrelated questions look similar. And a high score is **not** a guarantee. Test examples that are tricky, such as “Where should I put my lunchbox?” (lunch or lost-and-found?).

In [ ]:
test_questions = [
    "Where do I check out novels?",
    "I cannot find my backpack.",
    "What time can I eat?",
    "Can a robot finish my essay?",
    "How do I repair my bicycle?",
    "Where should I put my lunchbox?",
]
for text in test_questions:
    result = classify_intent(text)
    print(f"{text:<42} -> {result['intent']:<15} (similarity {result['score']:.2f})")

## 7. Check the mechanics
 The tests verify that our model returns vectors and labels. They do not certify that every prediction is accurate.

In [ ]:
vector = encoder.encode(["hello"], normalize_embeddings=True)
assert vector.shape[1] == 384
assert classify_intent("How can I borrow a book?")["intent"] in {"library", "lost_and_found", "lunch", "ai_policy", "unknown"}
assert len(set(label for _, label in examples)) == 4
assert np.isfinite(vector).all()
print("✅ Mission 2 checks passed!")

## Exit ticket
 1. What is an embedding? Do the numbers have obvious human-readable meanings?
 2. What does **intent classification** try to predict?
 3. Explain one mistake your classifier made or almost made.
 4. What is a reason to keep the **unknown** response?

 **Next:** In Mission 3, we find evidence in a mini handbook *before* preparing an AI answer.